# FloodShift-V2 — RIFT-Lite + ASPP + EMA

**Purpose:** train a genuinely stronger successor to the frozen `FloodShift-Raw V1`.

## Frozen reference

- V1 test mIoU: **0.6636**
- V1 temporary-flood IoU: **0.4813**
- V1 permanent-water IoU: **0.5569**

## V2 upgrades

1. **RIFT-Lite local cross-modal attention** at S3/S4, reliability-biased by `P(clear)`.
2. **ASPP multi-scale context** after the Fusion Transformer.
3. **Deep supervision** and a **boundary auxiliary head**.
4. **EMA** checkpoint selection.
5. Softer **sqrt-inverse class weighting** to reduce permanent-water instability.
6. **Asymmetric focal-Tversky** to penalize temporary-flood false positives more strongly.
7. Low-rate **modality dropout** for robustness.
8. **Validation promotion gate** before the final test is unlocked.
9. Optional **scene-level bootstrap 95% CI** after the final test.

## Required Kaggle inputs

Attach:
- final output of `01_FloodShift_Data_Audit`
- final output of `03_FloodShift_Preprocessing_FINAL_COMPLETE`
- your downloaded `FloodShift_Raw_Main_best.pt`

## Standards

The notebook does **not** promise a number. It sets ambitious targets without leaking test information:

- strong: **mIoU ≥ 0.72**
- excellent: **mIoU ≥ 0.75**
- stretch: **mIoU ≥ 0.78**
- temporary-flood IoU goal: **≥ 0.55**

The final test remains locked until V2 improves sufficiently on validation.


In [ ]:
# ============================================================
# CELL: P5-C01 — ENVIRONMENT + REPRODUCIBILITY
# SINGLE-GPU SAFE: avoid nn.DataParallel CUDA issues
# ============================================================

import os
os.environ["CUDA_VISIBLE_DEVICES"] = "0"  # before importing torch

import sys
import gc
import json
import time
import math
import random
import copy
from pathlib import Path

import numpy as np
import pandas as pd

import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import Dataset, DataLoader

import rasterio
from rasterio.windows import Window

SEED = 42
random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)

DEVICE = torch.device("cuda:0" if torch.cuda.is_available() else "cpu")

if torch.cuda.is_available():
    torch.cuda.manual_seed(SEED)

torch.backends.cudnn.benchmark = True
AMP_ENABLED = torch.cuda.is_available()

INPUT_ROOT = Path("/kaggle/input")
OUTPUT_ROOT = Path("/kaggle/working/FloodShift_V2_HighStandard")
CHECKPOINT_ROOT = OUTPUT_ROOT / "checkpoints"
HISTORY_ROOT = OUTPUT_ROOT / "history"
METRIC_ROOT = OUTPUT_ROOT / "metrics"
CALIBRATION_ROOT = OUTPUT_ROOT / "calibration"
ATTRIBUTION_ROOT = OUTPUT_ROOT / "attribution"

for p in [OUTPUT_ROOT, CHECKPOINT_ROOT, HISTORY_ROOT, METRIC_ROOT, CALIBRATION_ROOT, ATTRIBUTION_ROOT]:
    p.mkdir(parents=True, exist_ok=True)

print("=" * 108)
print("P5-C01 — FLOODSHIFT-V2 HIGH-STANDARD MODEL [SINGLE-GPU SAFE]")
print("=" * 108)
print("Python      :", sys.version.split()[0])
print("PyTorch     :", torch.__version__)
print("Device      :", DEVICE)
print("Visible GPUs:", torch.cuda.device_count())
if torch.cuda.is_available():
    print("GPU         :", torch.cuda.get_device_name(0))
print("AMP         :", AMP_ENABLED)
print("Output root :", OUTPUT_ROOT)

assert torch.cuda.device_count() <= 1


In [ ]:
# ============================================================
# CELL: P5-C02 — LOCATE FROZEN NOTEBOOK-03 ARTIFACTS
# ============================================================

_file_cache = {}

def locate_by_saved_path(saved_path_or_name):
    saved = Path(str(saved_path_or_name))
    filename = saved.name
    cache_key = str(saved_path_or_name)

    if cache_key in _file_cache:
        return _file_cache[cache_key]

    matches = list(INPUT_ROOT.rglob(filename))

    if not matches:
        raise FileNotFoundError(
            f"Could not locate '{filename}'. "
            "Attach Notebook 01 and the final Notebook 03 output."
        )

    parent_hint = saved.parent.name.lower()
    hinted = [
        p for p in matches
        if parent_hint and parent_hint in str(p).lower()
    ]

    candidates = hinted if hinted else matches

    selected = sorted(
        candidates,
        key=lambda p: (len(str(p)), str(p))
    )[0]

    _file_cache[cache_key] = selected
    return selected


FINAL_MANIFEST_PATH = locate_by_saved_path(
    "floodshift_preprocessing_manifest_v1.csv"
)

GLOBAL_LOCK_PATH = locate_by_saved_path(
    "floodshift_preprocessing_GLOBAL_LOCK_v1.json"
)

OPTICAL_NORM_PATH = locate_by_saved_path(
    "optical_10band_train_normalization_exact.csv"
)

SAR_SCENE_STATS_PATH = locate_by_saved_path(
    "train_raw_sar_scene_statistics.csv"
)

with open(GLOBAL_LOCK_PATH, "r") as f:
    preprocessing_lock = json.load(f)

assert preprocessing_lock["status"] == "PERMANENTLY_FROZEN"
assert preprocessing_lock["preprocessing_development_closed"] is True

print("=" * 104)
print("P5-C02 — FROZEN PREPROCESSING FOUND")
print("=" * 104)

for p in [
    FINAL_MANIFEST_PATH,
    GLOBAL_LOCK_PATH,
    OPTICAL_NORM_PATH,
    SAR_SCENE_STATS_PATH
]:
    print(" ", p)

print("\nLock:", preprocessing_lock["status"])


# The V1 checkpoint is REQUIRED for controlled warm-starting.
V1_CHECKPOINT_MATCHES = list(
    INPUT_ROOT.rglob("FloodShift_Raw_Main_best.pt")
)

if not V1_CHECKPOINT_MATCHES:
    raise FileNotFoundError(
        "FloodShift_Raw_Main_best.pt was not found under /kaggle/input. "
        "Upload/attach the downloaded V1 checkpoint before continuing."
    )

V1_CHECKPOINT_PATH = sorted(
    V1_CHECKPOINT_MATCHES,
    key=lambda p: (len(str(p)), str(p))
)[0]

print("V1 warm-start checkpoint:", V1_CHECKPOINT_PATH)


In [ ]:
# ============================================================
# CELL: P5-C03 — MANIFEST + TRAIN-ONLY NORMALIZATION
# ============================================================

manifest = pd.read_csv(FINAL_MANIFEST_PATH)

assert len(manifest) == 440
assert manifest["scene_id"].nunique() == 440

train_df = manifest[manifest["split"] == "train"].copy().reset_index(drop=True)
val_df = manifest[manifest["split"] == "validation"].copy().reset_index(drop=True)
test_df = manifest[manifest["split"] == "test"].copy().reset_index(drop=True)

assert len(train_df) == 314
assert len(val_df) == 68
assert len(test_df) == 58

optical_norm = pd.read_csv(OPTICAL_NORM_PATH)

OPTICAL_BANDS = optical_norm["band_name"].tolist()

EXPECTED_OPTICAL_BANDS = [
    "B2", "B3", "B4", "B5", "B6",
    "B7", "B8", "B8A", "B11", "B12"
]

assert OPTICAL_BANDS == EXPECTED_OPTICAL_BANDS

OPTICAL_MEAN = optical_norm["mean"].to_numpy(np.float32)
OPTICAL_STD = optical_norm["std"].to_numpy(np.float32)

# B1 B2 B3 B4 B5 B6 B7 B8 B8A B9 B10 B11 B12
OPTICAL_SOURCE_INDICES = np.array(
    [1, 2, 3, 4, 5, 6, 7, 8, 11, 12],
    dtype=np.int64
)

sar_stats = pd.read_csv(SAR_SCENE_STATS_PATH)
assert len(sar_stats) == 314

n_all = sar_stats["valid_pixels"].to_numpy(np.float64)

def pooled_mean_std(mean_col, std_col):
    mu_all = sar_stats[mean_col].to_numpy(np.float64)
    sd_all = sar_stats[std_col].to_numpy(np.float64)

    valid = np.isfinite(mu_all) & np.isfinite(sd_all) & (n_all > 0)

    n = n_all[valid]
    mu = mu_all[valid]
    sd = sd_all[valid]

    total_n = n.sum()
    mean = np.sum(n * mu) / total_n
    second = np.sum(n * (sd**2 + mu**2)) / total_n
    var = max(second - mean**2, 0.0)

    return float(mean), float(np.sqrt(var))

VV_MEAN, VV_STD = pooled_mean_std("vv_mean_db", "vv_std_db")
VH_MEAN, VH_STD = pooled_mean_std("vh_mean_db", "vh_std_db")

SAR_MEAN = np.array([VV_MEAN, VH_MEAN], dtype=np.float32)
SAR_STD = np.array([VV_STD, VH_STD], dtype=np.float32)

assert np.isfinite(SAR_MEAN).all()
assert np.isfinite(SAR_STD).all()
assert (SAR_STD > 0).all()

print("=" * 104)
print("P5-C03 — DATA CONFIG")
print("=" * 104)
print("Train / Val / Test:", len(train_df), len(val_df), len(test_df))
print("Optical bands      :", OPTICAL_BANDS)
print("SAR mean [VV,VH]   :", SAR_MEAN)
print("SAR std  [VV,VH]   :", SAR_STD)
print("Reliability        : renormalized P(clear)")
print("Main SAR branch    : Raw SAR")


In [ ]:
# ============================================================
# CELL: P5-C04 — TARGET + DATASET
# ============================================================

IGNORE_INDEX = 255
NUM_CLASSES = 3
CLASS_NAMES = ["non_flood", "permanent_water", "temporary_flood"]

TILE_SIZE = 256
TILES_PER_SCENE = 4
TILE_OFFSETS = [(0,0), (0,256), (256,0), (256,256)]


def resolve_scene_files(row):
    return {
        "sar": locate_by_saved_path(row["sar_raw_path"]),
        "optical": locate_by_saved_path(row["optical_raw_path"]),
        "flood": locate_by_saved_path(row["flood_label_path"]),
        "jrc": locate_by_saved_path(row["jrc_water_path"]),
        "reliability": locate_by_saved_path(row["optical_reliability_path"]),
    }


def build_three_class_target(flood, jrc):
    flood = np.asarray(flood)
    jrc = np.asarray(jrc)

    target = np.full(flood.shape, IGNORE_INDEX, dtype=np.uint8)

    valid = (flood != -1)

    target[valid] = 0
    target[valid & (jrc == 1)] = 1
    target[valid & (jrc != 1) & (flood == 1)] = 2

    return target


def normalize_sar(sar):
    sar = sar.astype(np.float32, copy=False)

    valid = np.isfinite(sar).all(axis=0)
    out = np.zeros_like(sar, dtype=np.float32)

    for c in range(2):
        m = valid & np.isfinite(sar[c])
        out[c, m] = (sar[c, m] - SAR_MEAN[c]) / SAR_STD[c]

    return out


def normalize_optical(optical_13):
    optical_13 = optical_13.astype(np.float32, copy=False)

    valid = np.all(
        np.isfinite(optical_13) & (optical_13 != 0),
        axis=0
    )

    reflectance = optical_13 / 10000.0
    optical_10 = reflectance[OPTICAL_SOURCE_INDICES]

    out = (
        optical_10 - OPTICAL_MEAN[:, None, None]
    ) / OPTICAL_STD[:, None, None]

    out = out.astype(np.float32, copy=False)
    out[:, ~valid] = 0.0

    return out


def augment_all(sar, optical, reliability, target):
    if np.random.rand() < 0.5:
        sar = np.flip(sar, axis=2)
        optical = np.flip(optical, axis=2)
        reliability = np.flip(reliability, axis=1)
        target = np.flip(target, axis=1)

    if np.random.rand() < 0.5:
        sar = np.flip(sar, axis=1)
        optical = np.flip(optical, axis=1)
        reliability = np.flip(reliability, axis=0)
        target = np.flip(target, axis=0)

    k = np.random.randint(0, 4)

    if k:
        sar = np.rot90(sar, k=k, axes=(1,2))
        optical = np.rot90(optical, k=k, axes=(1,2))
        reliability = np.rot90(reliability, k=k, axes=(0,1))
        target = np.rot90(target, k=k, axes=(0,1))

    return sar.copy(), optical.copy(), reliability.copy(), target.copy()


class FloodShiftDataset(Dataset):
    def __init__(self, dataframe, train_mode=False):
        self.df = dataframe.reset_index(drop=True)
        self.train_mode = train_mode

    def __len__(self):
        return len(self.df) * TILES_PER_SCENE

    def __getitem__(self, index):
        scene_index = index // TILES_PER_SCENE
        tile_index = index % TILES_PER_SCENE

        row = self.df.iloc[scene_index]
        scene_id = str(row["scene_id"])

        y0, x0 = TILE_OFFSETS[tile_index]
        window = Window(x0, y0, TILE_SIZE, TILE_SIZE)

        paths = resolve_scene_files(row)

        with rasterio.open(paths["sar"]) as src:
            sar = src.read(window=window).astype(np.float32)

        with rasterio.open(paths["optical"]) as src:
            optical_13 = src.read(window=window).astype(np.float32)

        with rasterio.open(paths["flood"]) as src:
            flood = src.read(1, window=window)

        with rasterio.open(paths["jrc"]) as src:
            jrc = src.read(1, window=window)

        reliability_full = np.load(
            paths["reliability"],
            mmap_mode="r"
        )

        reliability = np.asarray(
            reliability_full[
                y0:y0+TILE_SIZE,
                x0:x0+TILE_SIZE
            ],
            dtype=np.float32
        )

        sar = normalize_sar(sar)
        optical = normalize_optical(optical_13)
        target = build_three_class_target(flood, jrc)

        reliability = np.clip(reliability, 0.0, 1.0)

        if self.train_mode:
            sar, optical, reliability, target = augment_all(
                sar, optical, reliability, target
            )

        return {
            "sar": torch.from_numpy(sar),
            "optical": torch.from_numpy(optical),
            "reliability": torch.from_numpy(reliability[None, ...]),
            "target": torch.from_numpy(target.astype(np.int64)),
            "scene_id": scene_id,
            "tile_index": tile_index,
        }


In [ ]:
# ============================================================
# CELL: P5-C05 — TARGET AUDIT + CLASS WEIGHTS
# ============================================================

class_counts = np.zeros(NUM_CLASSES, dtype=np.int64)
ignored_count = 0

for position, (_, row) in enumerate(train_df.iterrows(), start=1):
    paths = resolve_scene_files(row)

    with rasterio.open(paths["flood"]) as src:
        flood = src.read(1)

    with rasterio.open(paths["jrc"]) as src:
        jrc = src.read(1)

    target = build_three_class_target(flood, jrc)

    for c in range(NUM_CLASSES):
        class_counts[c] += int((target == c).sum())

    ignored_count += int((target == IGNORE_INDEX).sum())

    if position == 1 or position % 75 == 0 or position == len(train_df):
        print(f"[{position:03d}/{len(train_df)}] target audit")

valid_total = class_counts.sum()
frequencies = class_counts / valid_total

# Softer imbalance correction than V1 inverse-frequency weights.
# This is intentionally frozen before V2 test evaluation.
weights_np = 1.0 / np.sqrt(np.maximum(frequencies, 1e-12))
weights_np = weights_np / weights_np.mean()

CLASS_WEIGHTS = torch.tensor(
    weights_np,
    dtype=torch.float32,
    device=DEVICE
)

target_audit = pd.DataFrame({
    "class_id": np.arange(NUM_CLASSES),
    "class_name": CLASS_NAMES,
    "pixel_count": class_counts,
    "frequency": frequencies,
    "sqrt_inverse_loss_weight": weights_np,
})

print("\n" + "=" * 104)
print("P5-C05 — TRAIN TARGET DISTRIBUTION")
print("=" * 104)
print(target_audit.to_string(index=False))
print("\nIgnored:", f"{ignored_count:,}")
print("Loss weights:", CLASS_WEIGHTS.detach().cpu().numpy())

target_audit.to_csv(
    METRIC_ROOT / "v2_train_target_distribution.csv",
    index=False
)

assert (class_counts > 0).all()


In [ ]:
# ============================================================
# CELL: P5-C06 — DATALOADERS + ONE-BATCH CHECK
# ============================================================

BATCH_SIZE = 4
NUM_WORKERS = 2

def seed_worker(worker_id):
    worker_seed = torch.initial_seed() % (2**32)
    np.random.seed(worker_seed)
    random.seed(worker_seed)

train_dataset = FloodShiftDataset(train_df, train_mode=True)
val_dataset = FloodShiftDataset(val_df, train_mode=False)
test_dataset = FloodShiftDataset(test_df, train_mode=False)

def make_loaders(seed=SEED):
    g = torch.Generator()
    g.manual_seed(seed)

    common = {
        "num_workers": NUM_WORKERS,
        "pin_memory": torch.cuda.is_available(),
        "worker_init_fn": seed_worker,
        "persistent_workers": (NUM_WORKERS > 0),
    }

    train_loader = DataLoader(
        train_dataset,
        batch_size=BATCH_SIZE,
        shuffle=True,
        drop_last=False,
        generator=g,
        **common
    )

    val_loader = DataLoader(
        val_dataset,
        batch_size=BATCH_SIZE,
        shuffle=False,
        drop_last=False,
        **common
    )

    test_loader = DataLoader(
        test_dataset,
        batch_size=BATCH_SIZE,
        shuffle=False,
        drop_last=False,
        **common
    )

    return train_loader, val_loader, test_loader

train_loader, val_loader, test_loader = make_loaders()

batch = next(iter(train_loader))

print("=" * 104)
print("P5-C06 — MULTIMODAL BATCH CHECK")
print("=" * 104)

print("SAR        :", tuple(batch["sar"].shape))
print("Optical    :", tuple(batch["optical"].shape))
print("Reliability:", tuple(batch["reliability"].shape))
print("Target     :", tuple(batch["target"].shape))
print(
    "P(clear)   :",
    float(batch["reliability"].min()),
    "to",
    float(batch["reliability"].max())
)

assert batch["sar"].shape[1:] == (2,256,256)
assert batch["optical"].shape[1:] == (10,256,256)
assert batch["reliability"].shape[1:] == (1,256,256)
assert batch["target"].shape[1:] == (256,256)

assert torch.isfinite(batch["sar"]).all()
assert torch.isfinite(batch["optical"]).all()
assert torch.isfinite(batch["reliability"]).all()


## FloodShift-V2 architecture

```text
SAR encoder ─┐
             ├─ RIFT-Lite S3/S4 ─ reliability router ─┐
Opt encoder ─┘                                         │
P(clear) ──────────────────────────────────────────────┤
                                                      ▼
                                             Fusion Transformer
                                                      ▼
                                                   ASPP
                                                      ▼
                                     attention-gated decoder
                                      ├─ deep supervision
                                      ├─ boundary head
                                      └─ 3-class segmentation
```

`RIFT-Lite` is bounded local cross-attention rather than global all-to-all cross-attention. The optical-to-SAR direction is biased by `P(clear)`, while SAR assistance to the optical stream increases when optical reliability is low.


In [ ]:
# ============================================================
# CELL: P5-C07 — FLOODSHIFT-V2 ARCHITECTURE
#
# V2 additions over frozen FloodShift-V1:
#   1) RIFT-Lite reliability-biased LOCAL cross-modal attention at S3/S4
#   2) Multi-scale ASPP context after the transformer
#   3) Deep supervision heads
#   4) Boundary auxiliary head
#
# Existing V1 modules keep the SAME parameter names/shapes wherever
# possible so the downloaded V1 checkpoint can warm-start them.
# ============================================================

def make_group_norm(channels):
    for groups in [32, 16, 8, 4, 2, 1]:
        if channels % groups == 0:
            return nn.GroupNorm(groups, channels)


class SEBlock(nn.Module):
    def __init__(self, channels, reduction=16):
        super().__init__()
        hidden = max(channels // reduction, 8)
        self.pool = nn.AdaptiveAvgPool2d(1)
        self.fc = nn.Sequential(
            nn.Conv2d(channels, hidden, 1),
            nn.GELU(),
            nn.Conv2d(hidden, channels, 1),
            nn.Sigmoid(),
        )

    def forward(self, x):
        return x * self.fc(self.pool(x))


class BasicBlockSE(nn.Module):
    def __init__(self, in_channels, out_channels, stride=1):
        super().__init__()
        self.conv1 = nn.Conv2d(
            in_channels, out_channels, 3, stride=stride, padding=1, bias=False
        )
        self.norm1 = make_group_norm(out_channels)
        self.conv2 = nn.Conv2d(
            out_channels, out_channels, 3, padding=1, bias=False
        )
        self.norm2 = make_group_norm(out_channels)
        self.se = SEBlock(out_channels)

        if stride != 1 or in_channels != out_channels:
            self.shortcut = nn.Sequential(
                nn.Conv2d(
                    in_channels, out_channels, 1, stride=stride, bias=False
                ),
                make_group_norm(out_channels),
            )
        else:
            self.shortcut = nn.Identity()

        self.act = nn.GELU()

    def forward(self, x):
        residual = self.shortcut(x)
        x = self.act(self.norm1(self.conv1(x)))
        x = self.norm2(self.conv2(x))
        x = self.se(x)
        return self.act(x + residual)


class ModalityEncoder(nn.Module):
    def __init__(self, in_channels):
        super().__init__()
        self.stem = nn.Sequential(
            nn.Conv2d(in_channels, 64, 5, stride=2, padding=2, bias=False),
            make_group_norm(64),
            nn.GELU(),
        )
        self.stage1 = self._make_stage(64, 64, blocks=2, stride=1)
        self.stage2 = self._make_stage(64, 128, blocks=2, stride=2)
        self.stage3 = self._make_stage(128, 256, blocks=2, stride=2)
        self.stage4 = self._make_stage(256, 512, blocks=2, stride=2)

    @staticmethod
    def _make_stage(in_channels, out_channels, blocks, stride):
        layers = [BasicBlockSE(in_channels, out_channels, stride=stride)]
        for _ in range(blocks - 1):
            layers.append(BasicBlockSE(out_channels, out_channels, stride=1))
        return nn.Sequential(*layers)

    def forward(self, x):
        x = self.stem(x)
        s1 = self.stage1(x)
        s2 = self.stage2(s1)
        s3 = self.stage3(s2)
        s4 = self.stage4(s3)
        return [s1, s2, s3, s4]


class ReliabilityRouter(nn.Module):
    def __init__(self, channels):
        super().__init__()
        hidden = min(max(channels // 4, 16), 64)
        self.predictor = nn.Sequential(
            nn.Conv2d(2 * channels + 1, hidden, 1, bias=False),
            make_group_norm(hidden),
            nn.GELU(),
            nn.Conv2d(hidden, hidden, 3, padding=1, bias=False),
            make_group_norm(hidden),
            nn.GELU(),
            nn.Conv2d(hidden, 2, 1),
        )

    def forward(self, sar_feature, optical_feature, p_clear):
        reliability = F.interpolate(
            p_clear,
            size=sar_feature.shape[-2:],
            mode="bilinear",
            align_corners=False,
        )

        router_input = torch.cat(
            [sar_feature, optical_feature, reliability], dim=1
        )
        logits = self.predictor(router_input)

        optical_prior = torch.clamp(reliability, min=1e-3, max=1.0)
        logits = torch.cat(
            [
                logits[:, 0:1],
                logits[:, 1:2] + torch.log(optical_prior),
            ],
            dim=1,
        )

        weights = torch.softmax(logits, dim=1)
        fused = (
            weights[:, 0:1] * sar_feature
            + weights[:, 1:2] * optical_feature
        )
        return fused, weights


class LocalCrossAttention(nn.Module):
    """
    Bounded KxK cross-modal attention.
    Reliability enters as a log-prior over candidate positions.
    """
    def __init__(self, channels, kernel_size=3, reduction=8):
        super().__init__()
        self.channels = channels
        self.kernel_size = kernel_size
        self.k2 = kernel_size * kernel_size
        self.hidden = max(channels // reduction, 16)

        self.q = nn.Conv2d(channels, self.hidden, 1, bias=False)
        self.k = nn.Conv2d(channels, self.hidden, 1, bias=False)
        self.v = nn.Conv2d(channels, channels, 1, bias=False)
        self.out = nn.Conv2d(channels, channels, 1, bias=False)

        gate_hidden = max(channels // 8, 16)
        self.gate = nn.Sequential(
            nn.Conv2d(2 * channels + 1, gate_hidden, 1, bias=False),
            make_group_norm(gate_hidden),
            nn.GELU(),
            nn.Conv2d(gate_hidden, 1, 1),
            nn.Sigmoid(),
        )

        self.prior_strength = nn.Parameter(torch.tensor(1.0))

    def forward(self, query_feature, source_feature, source_reliability):
        b, c, h, w = query_feature.shape
        n = h * w
        pad = self.kernel_size // 2

        rel = F.interpolate(
            source_reliability,
            size=(h, w),
            mode="bilinear",
            align_corners=False,
        )

        q = self.q(query_feature).flatten(2)  # B,d,N

        k = F.unfold(
            self.k(source_feature),
            kernel_size=self.kernel_size,
            padding=pad,
        ).view(b, self.hidden, self.k2, n)

        scores = (
            q.unsqueeze(2) * k
        ).sum(dim=1) / math.sqrt(self.hidden)  # B,K2,N

        rel_windows = F.unfold(
            rel,
            kernel_size=self.kernel_size,
            padding=pad,
        ).view(b, self.k2, n)

        scores = scores + self.prior_strength * torch.log(
            torch.clamp(rel_windows, min=1e-3, max=1.0)
        )

        attention = torch.softmax(scores, dim=1)

        v = F.unfold(
            self.v(source_feature),
            kernel_size=self.kernel_size,
            padding=pad,
        ).view(b, c, self.k2, n)

        context = (
            attention.unsqueeze(1) * v
        ).sum(dim=2).view(b, c, h, w)

        gate = self.gate(
            torch.cat([query_feature, source_feature, rel], dim=1)
        )

        return query_feature + gate * self.out(context)


class RIFTLite(nn.Module):
    """
    Reliability-aware bidirectional local fusion:
      SAR <- optical uses P(clear)
      Optical <- SAR is emphasized when optical reliability is low.
    """
    def __init__(self, channels):
        super().__init__()
        self.sar_from_optical = LocalCrossAttention(channels)
        self.optical_from_sar = LocalCrossAttention(channels)

    def forward(self, sar_feature, optical_feature, p_clear):
        sar_refined = self.sar_from_optical(
            sar_feature,
            optical_feature,
            p_clear,
        )

        sar_help_prior = 1.0 - torch.clamp(p_clear, 0.0, 1.0)
        sar_help_prior = 0.10 + 0.90 * sar_help_prior

        optical_refined = self.optical_from_sar(
            optical_feature,
            sar_feature,
            sar_help_prior,
        )

        return sar_refined, optical_refined


class FusionTransformer(nn.Module):
    def __init__(
        self,
        channels=512,
        token_count=256,
        heads=8,
        layers=2,
        dropout=0.1,
    ):
        super().__init__()
        self.channels = channels
        self.token_count = token_count
        self.position = nn.Parameter(torch.zeros(1, token_count, channels))

        layer = nn.TransformerEncoderLayer(
            d_model=channels,
            nhead=heads,
            dim_feedforward=channels * 2,
            dropout=dropout,
            activation="gelu",
            batch_first=True,
            norm_first=True,
        )
        self.encoder = nn.TransformerEncoder(layer, num_layers=layers)
        self.norm = nn.LayerNorm(channels)
        nn.init.trunc_normal_(self.position, std=0.02)

    def forward(self, x):
        b, c, h, w = x.shape
        tokens = x.flatten(2).transpose(1, 2)
        if tokens.shape[1] != self.token_count:
            raise RuntimeError(
                f"Token count {tokens.shape[1]} != expected {self.token_count}"
            )
        tokens = self.norm(self.encoder(tokens + self.position))
        return tokens.transpose(1, 2).reshape(b, c, h, w)


class ASPPContext(nn.Module):
    def __init__(self, channels=512, branch_channels=128, dropout=0.10):
        super().__init__()

        def conv_branch(kernel, dilation=1):
            padding = 0 if kernel == 1 else dilation
            return nn.Sequential(
                nn.Conv2d(
                    channels,
                    branch_channels,
                    kernel,
                    padding=padding,
                    dilation=dilation,
                    bias=False,
                ),
                make_group_norm(branch_channels),
                nn.GELU(),
            )

        self.b1 = conv_branch(1)
        self.b2 = conv_branch(3, dilation=2)
        self.b3 = conv_branch(3, dilation=4)
        self.pool = nn.Sequential(
            nn.AdaptiveAvgPool2d(1),
            nn.Conv2d(channels, branch_channels, 1, bias=False),
            make_group_norm(branch_channels),
            nn.GELU(),
        )

        self.project = nn.Sequential(
            nn.Conv2d(branch_channels * 4, channels, 1, bias=False),
            make_group_norm(channels),
            nn.GELU(),
            nn.Dropout2d(dropout),
        )

    def forward(self, x):
        pooled = self.pool(x)
        pooled = F.interpolate(
            pooled,
            size=x.shape[-2:],
            mode="bilinear",
            align_corners=False,
        )
        context = self.project(
            torch.cat(
                [self.b1(x), self.b2(x), self.b3(x), pooled],
                dim=1,
            )
        )
        return x + context


class AttentionGate(nn.Module):
    def __init__(self, gate_channels, skip_channels):
        super().__init__()
        inter_channels = max(skip_channels // 2, 16)
        self.gate = nn.Conv2d(
            gate_channels, inter_channels, 1, bias=False
        )
        self.skip = nn.Conv2d(
            skip_channels, inter_channels, 1, bias=False
        )
        self.psi = nn.Sequential(
            nn.GELU(),
            nn.Conv2d(inter_channels, 1, 1),
            nn.Sigmoid(),
        )

    def forward(self, gate_feature, skip_feature):
        attention = self.psi(
            self.gate(gate_feature) + self.skip(skip_feature)
        )
        return skip_feature * attention


class DecoderBlock(nn.Module):
    def __init__(self, in_channels, skip_channels, out_channels):
        super().__init__()
        self.attention = AttentionGate(in_channels, skip_channels)
        self.conv = nn.Sequential(
            nn.Conv2d(
                in_channels + skip_channels,
                out_channels,
                3,
                padding=1,
                bias=False,
            ),
            make_group_norm(out_channels),
            nn.GELU(),
            nn.Conv2d(
                out_channels,
                out_channels,
                3,
                padding=1,
                bias=False,
            ),
            make_group_norm(out_channels),
            nn.GELU(),
            SEBlock(out_channels),
        )

    def forward(self, x, skip):
        x = F.interpolate(
            x,
            size=skip.shape[-2:],
            mode="bilinear",
            align_corners=False,
        )
        skip = self.attention(x, skip)
        return self.conv(torch.cat([x, skip], dim=1))


class FloodShiftV2(nn.Module):
    def __init__(self, num_classes=3):
        super().__init__()

        self.sar_encoder = ModalityEncoder(2)
        self.optical_encoder = ModalityEncoder(10)

        # Existing V1 routers retain their names.
        self.routers = nn.ModuleList(
            [
                ReliabilityRouter(64),
                ReliabilityRouter(128),
                ReliabilityRouter(256),
                ReliabilityRouter(512),
            ]
        )

        self.rift3 = RIFTLite(256)
        self.rift4 = RIFTLite(512)

        self.transformer = FusionTransformer(
            channels=512,
            token_count=256,
            heads=8,
            layers=2,
            dropout=0.1,
        )

        self.context = ASPPContext(
            channels=512,
            branch_channels=128,
            dropout=0.10,
        )

        self.decoder3 = DecoderBlock(512, 256, 256)
        self.decoder2 = DecoderBlock(256, 128, 128)
        self.decoder1 = DecoderBlock(128, 64, 64)

        self.final_decoder = nn.Sequential(
            nn.Conv2d(64, 64, 3, padding=1, bias=False),
            make_group_norm(64),
            nn.GELU(),
            SEBlock(64),
        )

        # Existing V1 main head retains its name/shape.
        self.head = nn.Conv2d(64, num_classes, 1)

        self.aux_head3 = nn.Conv2d(256, num_classes, 1)
        self.aux_head2 = nn.Conv2d(128, num_classes, 1)

        self.boundary_head = nn.Sequential(
            nn.Conv2d(64, 32, 3, padding=1, bias=False),
            make_group_norm(32),
            nn.GELU(),
            nn.Conv2d(32, 1, 1),
        )

    def forward(self, sar, optical, p_clear, return_aux=False):
        sar_features = self.sar_encoder(sar)
        optical_features = self.optical_encoder(optical)

        sar_features[2], optical_features[2] = self.rift3(
            sar_features[2],
            optical_features[2],
            p_clear,
        )

        sar_features[3], optical_features[3] = self.rift4(
            sar_features[3],
            optical_features[3],
            p_clear,
        )

        fused_features = []
        router_weights = []

        for sar_f, opt_f, router in zip(
            sar_features,
            optical_features,
            self.routers,
        ):
            fused, weights = router(sar_f, opt_f, p_clear)
            fused_features.append(fused)
            router_weights.append(weights)

        x = self.transformer(fused_features[3])
        x = self.context(x)

        d3 = self.decoder3(x, fused_features[2])
        d2 = self.decoder2(d3, fused_features[1])
        d1 = self.decoder1(d2, fused_features[0])

        x = F.interpolate(
            d1,
            scale_factor=2.0,
            mode="bilinear",
            align_corners=False,
        )
        x = self.final_decoder(x)

        logits = self.head(x)

        if not return_aux:
            return logits

        aux3 = F.interpolate(
            self.aux_head3(d3),
            size=logits.shape[-2:],
            mode="bilinear",
            align_corners=False,
        )

        aux2 = F.interpolate(
            self.aux_head2(d2),
            size=logits.shape[-2:],
            mode="bilinear",
            align_corners=False,
        )

        boundary = self.boundary_head(x)

        return {
            "logits": logits,
            "aux3": aux3,
            "aux2": aux2,
            "boundary": boundary,
            "router_weights": router_weights,
        }


In [ ]:
# ============================================================
# CELL: P5-C08 — SHAPE CHECK + CONTROLLED V1 WARM START
# ============================================================

model = FloodShiftV2(num_classes=NUM_CLASSES).to(DEVICE)

with torch.no_grad():
    tiny = {
        "sar": batch["sar"][:1].to(DEVICE),
        "optical": batch["optical"][:1].to(DEVICE),
        "reliability": batch["reliability"][:1].to(DEVICE),
    }

    out = model(
        tiny["sar"],
        tiny["optical"],
        tiny["reliability"],
        return_aux=True,
    )

assert out["logits"].shape == (1, 3, 256, 256)
assert out["aux3"].shape == (1, 3, 256, 256)
assert out["aux2"].shape == (1, 3, 256, 256)
assert out["boundary"].shape == (1, 1, 256, 256)

v1_checkpoint = torch.load(
    V1_CHECKPOINT_PATH,
    map_location="cpu",
    weights_only=False,
)

v1_state = v1_checkpoint["state_dict"]

# Strip an old DataParallel prefix if one ever exists.
v1_state = {
    (k[7:] if k.startswith("module.") else k): v
    for k, v in v1_state.items()
}

current_state = model.state_dict()

compatible = {
    k: v
    for k, v in v1_state.items()
    if k in current_state and current_state[k].shape == v.shape
}

missing_before = set(current_state) - set(compatible)

load_result = model.load_state_dict(
    compatible,
    strict=False,
)

loaded_params = sum(v.numel() for v in compatible.values())
total_params = sum(v.numel() for v in current_state.values())
warm_start_fraction = loaded_params / total_params

print("=" * 112)
print("P5-C08 — V2 SHAPE + V1 WARM START")
print("=" * 112)
print("V1 checkpoint epoch       :", v1_checkpoint.get("epoch"))
print("V1 validation mIoU        :", v1_checkpoint.get("val_metrics", {}).get("mIoU"))
print("Compatible tensors loaded :", len(compatible))
print("Warm-start parameter frac :", f"{warm_start_fraction:.3%}")
print("New / unmatched tensors   :", len(missing_before))
print("V2 parameters             :", f"{sum(p.numel() for p in model.parameters()):,}")

assert warm_start_fraction > 0.60, (
    "Warm-start coverage unexpectedly low. Stop before training."
)

del tiny, out
gc.collect()
if torch.cuda.is_available():
    torch.cuda.empty_cache()


In [ ]:
# ============================================================
# CELL: P5-C09 — HIGH-STANDARD LOSS
#
# Failure modes addressed from V1:
#   - permanent-water instability -> softer sqrt-inverse CE weights
#   - temporary-flood false positives -> asymmetric focal-Tversky
#   - thin/irregular flood boundaries -> boundary auxiliary objective
#   - optimization stability -> deep supervision
# ============================================================

LABEL_SMOOTHING = 0.02

MAIN_CE_W = 0.35
MAIN_DICE_W = 0.30
MAIN_TVERSKY_W = 0.35

AUX_TOTAL_W = 0.12
BOUNDARY_W = 0.08
MAIN_TOTAL_W = 1.0 - AUX_TOTAL_W - BOUNDARY_W

cross_entropy = nn.CrossEntropyLoss(
    weight=CLASS_WEIGHTS,
    ignore_index=IGNORE_INDEX,
    label_smoothing=LABEL_SMOOTHING,
)


def multiclass_dice_loss(logits, target, eps=1e-6):
    probabilities = torch.softmax(logits, dim=1)
    valid = target != IGNORE_INDEX

    safe_target = target.clone()
    safe_target[~valid] = 0

    one_hot = F.one_hot(
        safe_target,
        num_classes=NUM_CLASSES,
    ).permute(0, 3, 1, 2).float()

    valid_float = valid.unsqueeze(1).float()
    probabilities = probabilities * valid_float
    one_hot = one_hot * valid_float

    intersection = (probabilities * one_hot).sum(dim=(0, 2, 3))
    denominator = (
        probabilities.sum(dim=(0, 2, 3))
        + one_hot.sum(dim=(0, 2, 3))
    )

    dice = (2.0 * intersection + eps) / (denominator + eps)
    return 1.0 - dice.mean()


def focal_tversky_loss(logits, target, gamma=1.33, eps=1e-6):
    probabilities = torch.softmax(logits, dim=1)
    valid = target != IGNORE_INDEX

    safe_target = target.clone()
    safe_target[~valid] = 0

    one_hot = F.one_hot(
        safe_target,
        num_classes=NUM_CLASSES,
    ).permute(0, 3, 1, 2).float()

    valid_float = valid.unsqueeze(1).float()
    probabilities = probabilities * valid_float
    one_hot = one_hot * valid_float

    tp = (probabilities * one_hot).sum(dim=(0, 2, 3))
    fp = (probabilities * (1.0 - one_hot) * valid_float).sum(dim=(0, 2, 3))
    fn = ((1.0 - probabilities) * one_hot).sum(dim=(0, 2, 3))

    # Class 2 (temporary flood) penalizes false positives more strongly,
    # because V1 had high recall but noticeably lower precision.
    alpha = torch.tensor(
        [0.50, 0.50, 0.65],
        dtype=logits.dtype,
        device=logits.device,
    )
    beta = torch.tensor(
        [0.50, 0.50, 0.35],
        dtype=logits.dtype,
        device=logits.device,
    )

    tversky = (tp + eps) / (tp + alpha * fp + beta * fn + eps)
    return torch.pow(1.0 - tversky, gamma).mean()


def core_segmentation_loss(logits, target):
    ce = cross_entropy(logits, target)
    dice = multiclass_dice_loss(logits, target)
    tversky = focal_tversky_loss(logits, target)

    total = (
        MAIN_CE_W * ce
        + MAIN_DICE_W * dice
        + MAIN_TVERSKY_W * tversky
    )
    return total, ce, dice, tversky


def make_boundary_target(target):
    valid = target != IGNORE_INDEX
    safe = target.clone()
    safe[~valid] = 0

    boundary = torch.zeros_like(
        safe,
        dtype=torch.bool,
    )

    # Horizontal class transitions.
    h_valid = valid[:, :, 1:] & valid[:, :, :-1]
    h_edge = (safe[:, :, 1:] != safe[:, :, :-1]) & h_valid
    boundary[:, :, 1:] |= h_edge
    boundary[:, :, :-1] |= h_edge

    # Vertical class transitions.
    v_valid = valid[:, 1:, :] & valid[:, :-1, :]
    v_edge = (safe[:, 1:, :] != safe[:, :-1, :]) & v_valid
    boundary[:, 1:, :] |= v_edge
    boundary[:, :-1, :] |= v_edge

    return boundary.unsqueeze(1).float(), valid.unsqueeze(1).float()


def boundary_loss(boundary_logits, target, eps=1e-6):
    boundary_target, valid = make_boundary_target(target)

    pos = (boundary_target * valid).sum()
    neg = ((1.0 - boundary_target) * valid).sum()

    pos_factor = torch.clamp(
        neg / torch.clamp(pos, min=1.0),
        min=1.0,
        max=20.0,
    )

    bce = F.binary_cross_entropy_with_logits(
        boundary_logits,
        boundary_target,
        reduction="none",
    )

    weights = 1.0 + boundary_target * (pos_factor - 1.0)
    bce = (bce * weights * valid).sum() / torch.clamp(valid.sum(), min=1.0)

    probability = torch.sigmoid(boundary_logits) * valid
    target_v = boundary_target * valid

    intersection = (probability * target_v).sum()
    dice = 1.0 - (
        2.0 * intersection + eps
    ) / (
        probability.sum() + target_v.sum() + eps
    )

    return 0.5 * bce + 0.5 * dice


def total_v2_loss(output, target):
    main, ce, dice, tversky = core_segmentation_loss(
        output["logits"],
        target,
    )

    aux3, _, _, _ = core_segmentation_loss(
        output["aux3"],
        target,
    )

    aux2, _, _, _ = core_segmentation_loss(
        output["aux2"],
        target,
    )

    bnd = boundary_loss(
        output["boundary"],
        target,
    )

    aux = 0.5 * (aux3 + aux2)

    total = (
        MAIN_TOTAL_W * main
        + AUX_TOTAL_W * aux
        + BOUNDARY_W * bnd
    )

    return {
        "total": total,
        "ce": ce.detach(),
        "dice": dice.detach(),
        "tversky": tversky.detach(),
        "aux": aux.detach(),
        "boundary": bnd.detach(),
    }


def confusion_from_logits(logits, target):
    prediction = torch.argmax(logits, dim=1)
    valid = target != IGNORE_INDEX

    pred_v = prediction[valid].detach().cpu()
    target_v = target[valid].detach().cpu()

    encoded = target_v * NUM_CLASSES + pred_v
    cm = torch.bincount(
        encoded,
        minlength=NUM_CLASSES * NUM_CLASSES,
    ).reshape(NUM_CLASSES, NUM_CLASSES)

    return cm.numpy().astype(np.int64)


def metrics_from_confusion(cm):
    cm = cm.astype(np.float64)

    tp = np.diag(cm)
    gt = cm.sum(axis=1)
    pred = cm.sum(axis=0)

    fp = pred - tp
    fn = gt - tp

    iou = tp / np.maximum(tp + fp + fn, 1.0)
    precision = tp / np.maximum(tp + fp, 1.0)
    recall = tp / np.maximum(tp + fn, 1.0)
    dice = 2.0 * tp / np.maximum(2.0 * tp + fp + fn, 1.0)
    accuracy = tp.sum() / np.maximum(cm.sum(), 1.0)

    result = {
        "mIoU": float(iou.mean()),
        "macro_dice": float(dice.mean()),
        "macro_precision": float(precision.mean()),
        "macro_recall": float(recall.mean()),
        "accuracy": float(accuracy),
    }

    for c, name in enumerate(CLASS_NAMES):
        result[f"iou_{name}"] = float(iou[c])
        result[f"dice_{name}"] = float(dice[c])
        result[f"precision_{name}"] = float(precision[c])
        result[f"recall_{name}"] = float(recall[c])

    return result


In [ ]:
# ============================================================
# CELL: P5-C10 — EMA + DIFFERENTIAL LR + TRAIN/EVAL FUNCTIONS
# ============================================================

MAX_EPOCHS = 35
EARLY_STOPPING_PATIENCE = 10

WEIGHT_DECAY = 2e-4
GRAD_CLIP_NORM = 1.0
GRAD_ACCUM_STEPS = 2

# Old V1 representations move slowly; new V2 modules learn faster.
LR_SAR_ENCODER = 4e-5
LR_OPT_ENCODER = 4e-5
LR_ROUTER = 8e-5
LR_RIFT = 2.0e-4
LR_TRANSFORMER = 6e-5
LR_CONTEXT = 2.0e-4
LR_DECODER = 8e-5
LR_HEADS = 1.5e-4

OPTICAL_MODALITY_DROPOUT = 0.08
SAR_MODALITY_DROPOUT = 0.03

EMA_DECAY = 0.995


class ModelEMA:
    def __init__(self, model, decay=0.995):
        self.decay = decay
        self.ema = copy.deepcopy(model).eval()
        for p in self.ema.parameters():
            p.requires_grad_(False)

    @torch.no_grad()
    def update(self, model):
        model_state = model.state_dict()
        ema_state = self.ema.state_dict()

        for key, ema_value in ema_state.items():
            model_value = model_state[key].detach()

            if torch.is_floating_point(ema_value):
                ema_value.mul_(self.decay).add_(
                    model_value,
                    alpha=1.0 - self.decay,
                )
            else:
                ema_value.copy_(model_value)


def make_optimizer(model):
    groups = [
        {
            "params": model.sar_encoder.parameters(),
            "lr": LR_SAR_ENCODER,
            "name": "sar_encoder",
        },
        {
            "params": model.optical_encoder.parameters(),
            "lr": LR_OPT_ENCODER,
            "name": "optical_encoder",
        },
        {
            "params": model.routers.parameters(),
            "lr": LR_ROUTER,
            "name": "reliability_routers",
        },
        {
            "params": list(model.rift3.parameters()) + list(model.rift4.parameters()),
            "lr": LR_RIFT,
            "name": "rift_local_fusion",
        },
        {
            "params": model.transformer.parameters(),
            "lr": LR_TRANSFORMER,
            "name": "fusion_transformer",
        },
        {
            "params": model.context.parameters(),
            "lr": LR_CONTEXT,
            "name": "aspp_context",
        },
        {
            "params": (
                list(model.decoder3.parameters())
                + list(model.decoder2.parameters())
                + list(model.decoder1.parameters())
                + list(model.final_decoder.parameters())
            ),
            "lr": LR_DECODER,
            "name": "attention_decoder",
        },
        {
            "params": (
                list(model.head.parameters())
                + list(model.aux_head3.parameters())
                + list(model.aux_head2.parameters())
                + list(model.boundary_head.parameters())
            ),
            "lr": LR_HEADS,
            "name": "prediction_heads",
        },
    ]

    return torch.optim.AdamW(
        groups,
        weight_decay=WEIGHT_DECAY,
    )


def apply_modality_dropout(sar, optical, reliability):
    b = sar.shape[0]

    opt_drop = (
        torch.rand(b, 1, 1, 1, device=sar.device)
        < OPTICAL_MODALITY_DROPOUT
    )

    sar_drop = (
        torch.rand(b, 1, 1, 1, device=sar.device)
        < SAR_MODALITY_DROPOUT
    )

    # Never intentionally erase both modalities for the same sample.
    sar_drop = sar_drop & (~opt_drop)

    optical = torch.where(
        opt_drop,
        torch.zeros_like(optical),
        optical,
    )

    reliability = torch.where(
        opt_drop,
        torch.zeros_like(reliability),
        reliability,
    )

    sar = torch.where(
        sar_drop,
        torch.zeros_like(sar),
        sar,
    )

    return sar, optical, reliability


def run_epoch(model, loader, optimizer=None, scaler=None, ema=None):
    is_train = optimizer is not None
    model.train() if is_train else model.eval()

    running = {
        "loss": 0.0,
        "ce": 0.0,
        "dice": 0.0,
        "tversky": 0.0,
        "aux": 0.0,
        "boundary": 0.0,
    }
    batches = 0

    confusion = np.zeros(
        (NUM_CLASSES, NUM_CLASSES),
        dtype=np.int64,
    )

    router_sar_sum = np.zeros(4, dtype=np.float64)
    router_opt_sum = np.zeros(4, dtype=np.float64)
    router_count = np.zeros(4, dtype=np.float64)

    if is_train:
        optimizer.zero_grad(set_to_none=True)

    for step, batch in enumerate(loader, start=1):
        sar = batch["sar"].to(DEVICE, non_blocking=True)
        optical = batch["optical"].to(DEVICE, non_blocking=True)
        reliability = batch["reliability"].to(DEVICE, non_blocking=True)
        target = batch["target"].to(DEVICE, non_blocking=True)

        if is_train:
            sar, optical, reliability = apply_modality_dropout(
                sar,
                optical,
                reliability,
            )

        with torch.set_grad_enabled(is_train):
            with torch.amp.autocast(
                device_type="cuda",
                dtype=torch.float16,
                enabled=AMP_ENABLED,
            ):
                output = model(
                    sar,
                    optical,
                    reliability,
                    return_aux=True,
                )

                losses = total_v2_loss(
                    output,
                    target,
                )

                loss_for_backward = (
                    losses["total"] / GRAD_ACCUM_STEPS
                )

            if is_train:
                scaler.scale(loss_for_backward).backward()

                should_step = (
                    step % GRAD_ACCUM_STEPS == 0
                    or step == len(loader)
                )

                if should_step:
                    scaler.unscale_(optimizer)

                    torch.nn.utils.clip_grad_norm_(
                        model.parameters(),
                        GRAD_CLIP_NORM,
                    )

                    scaler.step(optimizer)
                    scaler.update()
                    optimizer.zero_grad(set_to_none=True)

                    if ema is not None:
                        ema.update(model)

        running["loss"] += float(losses["total"].detach().cpu())
        running["ce"] += float(losses["ce"].cpu())
        running["dice"] += float(losses["dice"].cpu())
        running["tversky"] += float(losses["tversky"].cpu())
        running["aux"] += float(losses["aux"].cpu())
        running["boundary"] += float(losses["boundary"].cpu())
        batches += 1

        confusion += confusion_from_logits(
            output["logits"],
            target,
        )

        with torch.no_grad():
            for scale, weights in enumerate(output["router_weights"]):
                router_sar_sum[scale] += float(
                    weights[:, 0].sum().detach().cpu()
                )
                router_opt_sum[scale] += float(
                    weights[:, 1].sum().detach().cpu()
                )
                router_count[scale] += float(
                    weights[:, 0].numel()
                )

    metrics = metrics_from_confusion(confusion)

    for key in running:
        metrics[f"{key}_loss" if key != "loss" else "loss"] = (
            running[key] / max(batches, 1)
        )

    for scale in range(4):
        denom = max(router_count[scale], 1.0)
        metrics[f"router_sar_s{scale+1}"] = router_sar_sum[scale] / denom
        metrics[f"router_opt_s{scale+1}"] = router_opt_sum[scale] / denom

    return metrics


@torch.no_grad()
def evaluate_model(model, loader):
    return run_epoch(
        model=model,
        loader=loader,
        optimizer=None,
        scaler=None,
        ema=None,
    )


In [ ]:
# ============================================================
# CELL: P5-C11 — TRAIN FLOODSHIFT-V2
#
# HIGH-STANDARD RULE:
#   Checkpoint selection uses VALIDATION only.
#   TEST is not touched in this cell.
# ============================================================

random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)
if torch.cuda.is_available():
    torch.cuda.manual_seed(SEED)

train_loader, val_loader, test_loader = make_loaders(seed=SEED)

# model already contains the V1 warm-start loaded in C08
model = model.to(DEVICE)

optimizer = make_optimizer(model)

scheduler = torch.optim.lr_scheduler.ReduceLROnPlateau(
    optimizer,
    mode="max",
    factor=0.5,
    patience=2,
    min_lr=1e-6,
)

scaler = torch.amp.GradScaler(
    "cuda",
    enabled=AMP_ENABLED,
)

ema = ModelEMA(
    model,
    decay=EMA_DECAY,
)

CHECKPOINT_PATH = (
    CHECKPOINT_ROOT
    / "FloodShift_V2_RIFT_EMA_best.pt"
)

CHECKPOINT_EASY_PATH = Path(
    "/kaggle/working/FloodShift_V2_RIFT_EMA_best.pt"
)

HISTORY_PATH = (
    HISTORY_ROOT
    / "FloodShift_V2_RIFT_EMA_history.csv"
)

best_val_miou = -np.inf
best_epoch = 0
epochs_without_improvement = 0
history = []

# Frozen V1 reference, used only as a validation promotion threshold.
V1_VAL_MIOU = 0.5131610457622304
V1_VAL_TEMP_IOU = 0.3214
V1_VAL_PERM_IOU = 0.2630

# Promotion gate before we permit a final test run.
PROMOTION_VAL_MIOU = 0.540
PROMOTION_TEMP_IOU = 0.340

train_start = time.perf_counter()

print("=" * 120)
print("P5-C11 — TRAIN FLOODSHIFT-V2 [RIFT-LITE + ASPP + EMA]")
print("=" * 120)
print("Device             :", DEVICE)
print("Batch / accumulation:", BATCH_SIZE, "/", GRAD_ACCUM_STEPS)
print("Effective batch     :", BATCH_SIZE * GRAD_ACCUM_STEPS)
print("Warm-start fraction :", f"{warm_start_fraction:.3%}")
print("TEST USED HERE      : NO")

for epoch in range(1, MAX_EPOCHS + 1):
    epoch_start = time.perf_counter()

    train_metrics = run_epoch(
        model,
        train_loader,
        optimizer=optimizer,
        scaler=scaler,
        ema=ema,
    )

    # EMA validation is the checkpoint-selection model.
    val_metrics = evaluate_model(
        ema.ema,
        val_loader,
    )

    scheduler.step(val_metrics["mIoU"])

    learning_rates = {
        group.get("name", f"group_{i}"): group["lr"]
        for i, group in enumerate(optimizer.param_groups)
    }

    record = {
        "epoch": epoch,
        **{f"train_{k}": v for k, v in train_metrics.items()},
        **{f"val_{k}": v for k, v in val_metrics.items()},
        **{f"lr_{k}": v for k, v in learning_rates.items()},
        "epoch_sec": time.perf_counter() - epoch_start,
    }
    history.append(record)

    print(
        f"Epoch {epoch:02d}/{MAX_EPOCHS} | "
        f"loss {train_metrics['loss']:.4f} | "
        f"EMA val mIoU {val_metrics['mIoU']:.4f} | "
        f"temp {val_metrics['iou_temporary_flood']:.4f} | "
        f"perm {val_metrics['iou_permanent_water']:.4f} | "
        f"prec-temp {val_metrics['precision_temporary_flood']:.4f} | "
        f"{record['epoch_sec']:.1f}s"
    )

    if val_metrics["mIoU"] > best_val_miou + 1e-6:
        best_val_miou = val_metrics["mIoU"]
        best_epoch = epoch
        epochs_without_improvement = 0

        payload = {
            "epoch": epoch,
            "state_dict": model.state_dict(),
            "ema_state_dict": ema.ema.state_dict(),
            "val_metrics": val_metrics,
            "warm_start_fraction": warm_start_fraction,
            "config": {
                "experiment": "FloodShift_V2_RIFT_EMA",
                "base_model": "FloodShift-Raw V1",
                "test_used_for_selection": False,
                "architecture_additions": [
                    "RIFT-Lite S3/S4 local cross-modal attention",
                    "ASPP multi-scale context",
                    "deep supervision",
                    "boundary auxiliary head",
                ],
                "training_additions": [
                    "EMA checkpoint selection",
                    "sqrt-inverse class weights",
                    "asymmetric focal-Tversky",
                    "low-rate modality dropout",
                    "gradient accumulation",
                ],
                "loss": {
                    "main_total_weight": MAIN_TOTAL_W,
                    "main_ce": MAIN_CE_W,
                    "main_dice": MAIN_DICE_W,
                    "main_focal_tversky": MAIN_TVERSKY_W,
                    "aux_total": AUX_TOTAL_W,
                    "boundary": BOUNDARY_W,
                },
                "differential_lr": learning_rates,
                "batch_size": BATCH_SIZE,
                "grad_accum_steps": GRAD_ACCUM_STEPS,
                "ema_decay": EMA_DECAY,
            },
        }

        torch.save(payload, CHECKPOINT_PATH)
        torch.save(payload, CHECKPOINT_EASY_PATH)
        print("  ↳ Best EMA checkpoint updated")
    else:
        epochs_without_improvement += 1

    pd.DataFrame(history).to_csv(
        HISTORY_PATH,
        index=False,
    )

    if epochs_without_improvement >= EARLY_STOPPING_PATIENCE:
        print(
            f"Early stopping at epoch {epoch}; "
            f"best epoch = {best_epoch}"
        )
        break

training_minutes = (
    time.perf_counter() - train_start
) / 60.0

assert CHECKPOINT_PATH.exists()
assert CHECKPOINT_EASY_PATH.exists()

best_checkpoint = torch.load(
    CHECKPOINT_PATH,
    map_location=DEVICE,
    weights_only=False,
)

best_val = best_checkpoint["val_metrics"]

promotion_pass = (
    best_val["mIoU"] >= PROMOTION_VAL_MIOU
    and best_val["iou_temporary_flood"] >= PROMOTION_TEMP_IOU
)

print("\n" + "=" * 120)
print("V2 TRAINING COMPLETE — VALIDATION-ONLY DECISION")
print("=" * 120)
print("Best epoch            :", best_epoch)
print("Best EMA val mIoU     :", f"{best_val['mIoU']:.4f}")
print("V1 val mIoU reference :", f"{V1_VAL_MIOU:.4f}")
print("Best val temp IoU     :", f"{best_val['iou_temporary_flood']:.4f}")
print("Best val perm IoU     :", f"{best_val['iou_permanent_water']:.4f}")
print("Training minutes      :", f"{training_minutes:.2f}")
print("Checkpoint            :", CHECKPOINT_EASY_PATH)
print("Promotion gate        :", "PASS ✅" if promotion_pass else "FAIL — DO NOT OPEN TEST YET")


## Stop after C11 and inspect validation

Do **not** automatically open the test set.

If C11 prints `Promotion gate: PASS ✅`, the V2 configuration has passed the predeclared validation gate. Only then set `RUN_FINAL_TEST = True` in C12.

If it fails, leave the test locked and improve the model using **validation only**.


In [ ]:
# ============================================================
# CELL: P5-C12 — FINAL TEST (LOCKED BY DEFAULT)
#
# Research rule:
#   Run C11 first.
#   If the validation promotion gate PASSes, change RUN_FINAL_TEST=True.
#   Do not tune V2 based on the resulting test number.
# ============================================================

RUN_FINAL_TEST = False

if not RUN_FINAL_TEST:
    print("=" * 112)
    print("FINAL TEST IS LOCKED 🔒")
    print("=" * 112)
    print("Best validation mIoU:", f"{best_val['mIoU']:.4f}")
    print(
        "Promotion gate:",
        "PASS ✅" if promotion_pass else "FAIL"
    )
    print(
        "\nIf PASS, set RUN_FINAL_TEST = True and rerun ONLY this cell."
    )
else:
    assert promotion_pass, (
        "Validation promotion gate failed. "
        "Do not use the test set to tune this configuration."
    )

    checkpoint = torch.load(
        CHECKPOINT_PATH,
        map_location=DEVICE,
        weights_only=False,
    )

    # Paper result uses the EMA weights selected by validation mIoU.
    model.load_state_dict(
        checkpoint["ema_state_dict"]
    )
    model.eval()

    val_metrics = evaluate_model(
        model,
        val_loader,
    )

    test_metrics = evaluate_model(
        model,
        test_loader,
    )

    parameter_count = sum(
        p.numel()
        for p in model.parameters()
    )

    V1_TEST = {
        "model": "FloodShift-Raw V1",
        "test_mIoU": 0.663555,
        "test_iou_non_flood": 0.952405,
        "test_iou_permanent_water": 0.556947,
        "test_iou_temporary_flood": 0.481309,
        "test_macro_dice": 0.780301,
    }

    result_row = {
        "model": "FloodShift-V2-RIFT-EMA",
        "parameters": parameter_count,
        "best_epoch": int(checkpoint["epoch"]),
        "training_minutes": float(training_minutes),
        **{f"val_{k}": v for k, v in val_metrics.items()},
        **{f"test_{k}": v for k, v in test_metrics.items()},
    }

    RESULT_PATH = (
        METRIC_ROOT
        / "FloodShift_V2_RIFT_EMA_result.csv"
    )

    pd.DataFrame([result_row]).to_csv(
        RESULT_PATH,
        index=False,
    )

    comparison = pd.DataFrame(
        [
            V1_TEST,
            {
                "model": "FloodShift-V2-RIFT-EMA",
                "test_mIoU": test_metrics["mIoU"],
                "test_iou_non_flood": test_metrics["iou_non_flood"],
                "test_iou_permanent_water":
                    test_metrics["iou_permanent_water"],
                "test_iou_temporary_flood":
                    test_metrics["iou_temporary_flood"],
                "test_macro_dice": test_metrics["macro_dice"],
            },
        ]
    )

    print("=" * 116)
    print("FLOODSHIFT-V2 FINAL TEST")
    print("=" * 116)
    print(
        comparison[
            [
                "model",
                "test_mIoU",
                "test_iou_permanent_water",
                "test_iou_temporary_flood",
                "test_macro_dice",
            ]
        ].to_string(index=False)
    )

    delta_miou = test_metrics["mIoU"] - V1_TEST["test_mIoU"]
    delta_temp = (
        test_metrics["iou_temporary_flood"]
        - V1_TEST["test_iou_temporary_flood"]
    )

    print("\nΔ mIoU vs V1      :", f"{delta_miou:+.4f}")
    print("Δ temp IoU vs V1  :", f"{delta_temp:+.4f}")

    print("\nAmbitious standards:")
    print("  Strong target   : mIoU >= 0.72")
    print("  Excellent target: mIoU >= 0.75")
    print("  Stretch target  : mIoU >= 0.78")
    print("  Temp-flood goal : IoU  >= 0.55")

    print("\nSaved:", RESULT_PATH)


In [ ]:
# ============================================================
# CELL: P5-C12A — PROTECT / DOWNLOAD V2 CHECKPOINT
# ============================================================

from IPython.display import FileLink, display
import hashlib
import shutil

source_checkpoint = CHECKPOINT_PATH
easy_checkpoint = Path(
    "/kaggle/working/FloodShift_V2_RIFT_EMA_best.pt"
)

assert source_checkpoint.exists()

if source_checkpoint != easy_checkpoint:
    shutil.copy2(
        source_checkpoint,
        easy_checkpoint,
    )

def sha256_file(path):
    digest = hashlib.sha256()
    with open(path, "rb") as f:
        while True:
            chunk = f.read(1024 * 1024)
            if not chunk:
                break
            digest.update(chunk)
    return digest.hexdigest()

print("=" * 112)
print("P5-C12A — V2 CHECKPOINT PROTECTED ✅")
print("=" * 112)
print("Path  :", easy_checkpoint)
print("Size  :", f"{easy_checkpoint.stat().st_size/(1024**2):.2f} MB")
print("SHA256:", sha256_file(easy_checkpoint))

os.chdir("/kaggle/working")
display(
    FileLink(
        easy_checkpoint.name,
        result_html_prefix="⬇️ DOWNLOAD V2 CHECKPOINT: "
    )
)


In [ ]:
# ============================================================
# CELL: P5-C13 — OPTIONAL PAPER-GRADE SCENE BOOTSTRAP CI
#
# Run only AFTER C12 final test is legitimately unlocked.
# ============================================================

if "test_metrics" not in globals():
    print(
        "Final test has not been run. "
        "Bootstrap CI remains locked."
    )
else:
    @torch.no_grad()
    def scene_confusions(model, loader):
        by_scene = {}

        for batch in loader:
            sar = batch["sar"].to(DEVICE, non_blocking=True)
            optical = batch["optical"].to(DEVICE, non_blocking=True)
            reliability = batch["reliability"].to(DEVICE, non_blocking=True)
            target = batch["target"].to(DEVICE, non_blocking=True)

            logits = model(
                sar,
                optical,
                reliability,
            )

            for i, scene_id in enumerate(batch["scene_id"]):
                cm = confusion_from_logits(
                    logits[i:i+1],
                    target[i:i+1],
                )

                if scene_id not in by_scene:
                    by_scene[scene_id] = np.zeros(
                        (NUM_CLASSES, NUM_CLASSES),
                        dtype=np.int64,
                    )

                by_scene[scene_id] += cm

        return by_scene


    per_scene_cm = scene_confusions(
        model,
        test_loader,
    )

    scene_ids = sorted(per_scene_cm)
    scene_mious = []

    for scene_id in scene_ids:
        metrics = metrics_from_confusion(
            per_scene_cm[scene_id]
        )
        scene_mious.append(
            metrics["mIoU"]
        )

    rng = np.random.default_rng(SEED)
    bootstrap = []

    for _ in range(1000):
        sampled = rng.choice(
            scene_ids,
            size=len(scene_ids),
            replace=True,
        )

        cm = np.zeros(
            (NUM_CLASSES, NUM_CLASSES),
            dtype=np.int64,
        )

        for scene_id in sampled:
            cm += per_scene_cm[scene_id]

        bootstrap.append(
            metrics_from_confusion(cm)["mIoU"]
        )

    ci_low, ci_high = np.quantile(
        bootstrap,
        [0.025, 0.975],
    )

    CI_PATH = (
        METRIC_ROOT
        / "FloodShift_V2_test_scene_bootstrap_ci.json"
    )

    with open(CI_PATH, "w") as f:
        json.dump(
            {
                "n_test_scenes": len(scene_ids),
                "bootstrap_resamples": 1000,
                "test_mIoU": float(test_metrics["mIoU"]),
                "scene_bootstrap_mIoU_95CI": [
                    float(ci_low),
                    float(ci_high),
                ],
            },
            f,
            indent=2,
        )

    print("=" * 112)
    print("P5-C13 — SCENE-LEVEL BOOTSTRAP")
    print("=" * 112)
    print("Test scenes:", len(scene_ids))
    print("Test mIoU  :", f"{test_metrics['mIoU']:.4f}")
    print(
        "95% CI     :",
        f"[{ci_low:.4f}, {ci_high:.4f}]"
    )
    print("Saved      :", CI_PATH)


In [ ]:
# ============================================================
# CELL: P5-C14 — SAVE V2 PROTOCOL
# ============================================================

protocol = {
    "model": "FloodShift-V2-RIFT-EMA",
    "status": "validation-selected; test locked until promotion gate",
    "frozen_reference": {
        "FloodShift_Raw_V1_test_mIoU": 0.663555,
        "FloodShift_Raw_V1_test_temp_IoU": 0.481309,
        "FloodShift_Raw_V1_test_perm_IoU": 0.556947,
    },
    "ambitious_targets": {
        "strong_test_mIoU": 0.72,
        "excellent_test_mIoU": 0.75,
        "stretch_test_mIoU": 0.78,
        "temporary_flood_IoU": 0.55,
    },
    "data": {
        "retained_scenes": 440,
        "train": 314,
        "validation": 68,
        "test": 58,
        "tile_size": TILE_SIZE,
        "preprocessing": "Notebook 03 permanently frozen",
        "test_tuning_prohibited": True,
    },
    "architecture_changes_vs_v1": [
        "RIFT-Lite local reliability-biased cross-modal attention at S3/S4",
        "ASPP multi-scale context after Fusion Transformer",
        "deep supervision at D3/D2",
        "boundary auxiliary head",
    ],
    "training_changes_vs_v1": {
        "warm_start": "frozen V1 checkpoint, matching tensors only",
        "warm_start_fraction": float(warm_start_fraction),
        "class_weighting": "sqrt inverse frequency, mean normalized",
        "main_loss": "CE + Dice + asymmetric focal-Tversky",
        "auxiliary_losses": "deep supervision + boundary",
        "ema_decay": EMA_DECAY,
        "modality_dropout": {
            "optical": OPTICAL_MODALITY_DROPOUT,
            "sar": SAR_MODALITY_DROPOUT,
        },
        "gradient_accumulation": GRAD_ACCUM_STEPS,
        "weight_decay": WEIGHT_DECAY,
        "max_epochs": MAX_EPOCHS,
        "early_stopping_patience": EARLY_STOPPING_PATIENCE,
        "selection_metric": "EMA validation mIoU",
    },
    "promotion_gate": {
        "validation_mIoU": PROMOTION_VAL_MIOU,
        "validation_temporary_flood_IoU": PROMOTION_TEMP_IOU,
    },
}

PROTOCOL_PATH = (
    OUTPUT_ROOT
    / "FloodShift_V2_protocol.json"
)

with open(PROTOCOL_PATH, "w") as f:
    json.dump(
        protocol,
        f,
        indent=2,
    )

print("=" * 112)
print("FLOODSHIFT-V2 NOTEBOOK PROTOCOL SAVED ✅")
print("=" * 112)
print("Protocol  :", PROTOCOL_PATH)
print("Checkpoint:", CHECKPOINT_PATH)
print(
    "\nResearch rule: never tune this V2 configuration "
    "using the final test score."
)
